In [ ]:
# =====================================
# 0. 라이브러리 불러오기
# =====================================
import pandas as pd
import re   # 텍스트 정제용

# 데이터 나누기
from sklearn.model_selection import train_test_split

# 텍스트 → 숫자로 변환
from sklearn.feature_extraction.text import TfidfVectorizer

# 오늘 사용할 모델 (SGDClassifier)
from sklearn.linear_model import SGDClassifier

# 성능 평가
from sklearn.metrics import accuracy_score


# =====================================
# 1. 데이터 불러오기
# =====================================
df = pd.read_csv("mbti_1.csv")

print("데이터 크기:", df.shape)
print(df.head())


# =====================================
# 2. 텍스트 전처리 함수 만들기
# =====================================
# 머신러닝은 깨끗한 데이터를 좋아함
# → 불필요한 요소 제거

def clean_text(text):
    # 1. URL 제거 (유튜브 링크 등)
    text = re.sub(r'http\S+', '', text)

    # 2. 영어 알파벳만 남기기 (숫자, 특수문자 제거)
    text = re.sub(r'[^a-zA-Z\s]', '', text)

    # 3. 소문자로 변환 (Love → love)
    text = text.lower()

    return text


# 모든 데이터에 적용
df['posts'] = df['posts'].apply(clean_text)


# =====================================
# 3. 입력(X) / 정답(y) 나누기
# =====================================
# X = 우리가 분석할 데이터 (텍스트)
# y = 맞춰야 할 정답 (MBTI)
X = df['posts']
y = df['type']


# =====================================
# 4. Train / Test 분리
# =====================================
# test_size=0.2 → 20%는 테스트용
# random_state=42 → 항상 같은 결과 나오게 고정 (중요)
# stratify=y → MBTI 비율 유지
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


# =====================================
# 5. Train / Validation 분리
# =====================================
# train 데이터를 다시 나눠서 검증 데이터 생성
# 최종 비율: train 60% / valid 20% / test 20%
X_train, X_valid, y_train, y_valid = train_test_split(
    X_train, y_train,
    test_size=0.25,
    random_state=42,
    stratify=y_train
)

print("\n데이터 분리 결과")
print("Train:", len(X_train))
print("Valid:", len(X_valid))
print("Test :", len(X_test))


# =====================================
# 6. 텍스트 → 숫자로 변환 (TF-IDF)
# =====================================
# 머신러닝은 숫자만 이해하기 때문에
# 텍스트를 숫자로 바꿔야 함

vectorizer = TfidfVectorizer(
    max_features=5000,     # 중요한 단어 5000개만 사용
    stop_words='english',  # 의미 없는 단어 제거 (the, is 등)
    ngram_range=(1, 2)     # 단어 1개 + 2개 묶음까지 사용
)

# ⭐ 매우 중요
# fit = 단어를 학습
# transform = 숫자로 변환

# 학습 데이터로만 fit 해야 함 (데이터 누수 방지)
X_train_vec = vectorizer.fit_transform(X_train)

# 검증 / 테스트는 transform만
X_valid_vec = vectorizer.transform(X_valid)
X_test_vec = vectorizer.transform(X_test)


# =====================================
# 7. 모델 생성 (SGDClassifier)
# =====================================
# SGD = 데이터를 조금씩 보면서 학습하는 방식
# → 속도가 빠르고 텍스트 데이터에 강함

model = SGDClassifier(
    loss='hinge',       # SVM 방식 (분류 문제에서 많이 사용)
    max_iter=1000,      # 반복 횟수
    random_state=42     # 결과 재현성
)


# =====================================
# 8. 모델 학습
# =====================================
model.fit(X_train_vec, y_train)


# =====================================
# 9. 검증 데이터로 성능 확인
# =====================================
# 모델이 얼마나 잘 맞추는지 확인

y_valid_pred = model.predict(X_valid_vec)

valid_acc = accuracy_score(y_valid, y_valid_pred)

print("\n검증 정확도:", valid_acc)


# =====================================
# 10. 테스트 데이터로 최종 평가
# =====================================
y_test_pred = model.predict(X_test_vec)

test_acc = accuracy_score(y_test, y_test_pred)

print("테스트 정확도:", test_acc)


# =====================================
# 11. 새로운 문장 예측해보기 (재미용🔥)
# =====================================
new_text = ["I enjoy deep conversations and thinking alone"]

# 같은 방식으로 변환
new_vec = vectorizer.transform(new_text)

# 예측
print("\n새 문장 예측:", model.predict(new_vec))


In [ ]:
# DataFrame으로 보기
# 벡터 → DataFrame으로 변환
df_tfidf = pd.DataFrame(
    X_train_vec.toarray(),                   # 숫자 데이터
    columns=vectorizer.get_feature_names_out()  # 단어 리스트
)

# 상위 5개 데이터 보기
print(df_tfidf.head())

In [ ]:
# 특정 문장만 보기
# # 첫 번째 문장 벡터 확인
first_vec = X_train_vec[0].toarray()[0]

# 단어 + 값 출력
for word, score in zip(vectorizer.get_feature_names_out(), first_vec):
    if score > 0:   # 값이 있는 단어만 출력
        print(word, ":", score)

In [ ]:
# 상위 중요 단어만 보기
import numpy as np

first_vec = X_train_vec[0].toarray()[0]

# 상위 10개 단어 뽑기
top_indices = np.argsort(first_vec)[-10:]

for idx in top_indices:
    print(vectorizer.get_feature_names_out()[idx], ":", first_vec[idx])

In [8]:
import pandas as pd
import re
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

df = pd.read_csv('data/mbti_1.csv')

print( df.shape )
print( df['type'].value_counts() )
print( df['posts'][0][:300] )

(8675, 2)
type
INFP    1832
INFJ    1470
INTP    1304
INTJ    1091
ENTP     685
ENFP     675
ISTP     337
ISFP     271
ENTJ     231
ISTJ     205
ENFJ     190
ISFJ     166
ESTP      89
ESFP      48
ESFJ      42
ESTJ      39
Name: count, dtype: int64
'http://www.youtube.com/watch?v=qsXHcwe3krw|||http://41.media.tumblr.com/tumblr_lfouy03PMA1qa1rooo1_500.jpg|||enfp and intj moments  https://www.youtube.com/watch?v=iz7lE1g4XM4  sportscenter not top ten plays  https://www.youtube.com/watch?v=uCdfze1etec  pranks|||What has been the most life-changing


In [64]:
# 데이터 전처리 함수 (url제거, 알파벳만 남긴 후 소문자로 변환)
def clean_text(text):
    text = re.sub(r'http\S+', '', text) # text에서 http로 시작하는 공백 없는 문자열을 ''로 바꾸기
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    text = text.lower()

    return text

In [65]:
df['posts'] = df['posts'].apply(clean_text) # apply : 모든 데이터에 clean_text 일괄 적용

X = df['posts']
y = df['type']

# 데이터 나누기 8:2
x_train,x_test,y_train,y_test = train_test_split( X, y, test_size=0.2, random_state=42, stratify=y )
# 데이터 train/valid로 나누기
x_train,x_valid,y_train,y_valid = train_test_split( x_train, y_train, test_size=0.25, random_state=42, stratify=y_train)

print("Train:", len(x_train))
print("Valid:", len(x_valid))
print("Test :", len(x_test))
# TF - IDF
# TF (Term Frequency) => 문서에서 얼마나 자주 나오냐
# IDE (Inverse Document Frequency) : 전체 문서에서 얼마나 희귀하냐
vectorizer = TfidfVectorizer(stop_words='english', max_features=5000, ngram_range=(1,2))
# stop_words : it, the 같은 쓸모없는 단어 제거
# max_features : 중요 단어 5000개만 사용
# ngram_range : 단어 1, 2 묶음 모두 사용
x_train_trans = vectorizer.fit_transform(x_train)
x_valid_trans = vectorizer.transform(x_valid)
x_test_trans = vectorizer.transform(x_test)

model = LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced')
# class_weight='balanced' : 데이터가 적은 유형에 더 높은 가중치
model.fit(x_train_trans, y_train)
y_pred = model.predict(x_valid_trans)
print(accuracy_score(y_valid, y_pred))
print(classification_report(y_valid, y_pred))




Train: 5205
Valid: 1735
Test : 1735
0.6518731988472622
              precision    recall  f1-score   support

        ENFJ       0.50      0.55      0.53        38
        ENFP       0.65      0.65      0.65       135
        ENTJ       0.47      0.57      0.51        46
        ENTP       0.68      0.67      0.68       137
        ESFJ       0.33      0.50      0.40         8
        ESFP       0.22      0.22      0.22         9
        ESTJ       0.75      0.38      0.50         8
        ESTP       0.43      0.56      0.49        18
        INFJ       0.75      0.60      0.67       294
        INFP       0.75      0.71      0.73       367
        INTJ       0.67      0.68      0.68       218
        INTP       0.64      0.69      0.66       261
        ISFJ       0.61      0.58      0.59        33
        ISFP       0.51      0.59      0.55        54
        ISTJ       0.48      0.61      0.54        41
        ISTP       0.50      0.66      0.57        68

    accuracy             

In [67]:
model = LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced')
model.fit(x_train_trans, y_train)
y_pred = model.predict(x_test_trans)
print(accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

0.6472622478386167
              precision    recall  f1-score   support

        ENFJ       0.51      0.63      0.56        38
        ENFP       0.64      0.59      0.61       135
        ENTJ       0.40      0.57      0.47        46
        ENTP       0.60      0.61      0.60       137
        ESFJ       0.27      0.33      0.30         9
        ESFP       0.00      0.00      0.00        10
        ESTJ       0.40      0.25      0.31         8
        ESTP       0.42      0.56      0.48        18
        INFJ       0.77      0.63      0.69       294
        INFP       0.75      0.70      0.72       366
        INTJ       0.68      0.64      0.66       218
        INTP       0.70      0.73      0.72       261
        ISFJ       0.50      0.55      0.52        33
        ISFP       0.43      0.54      0.48        54
        ISTJ       0.44      0.61      0.51        41
        ISTP       0.57      0.76      0.65        67

    accuracy                           0.65      1735
   macr

In [70]:
x_train,x_test,y_train,y_test = train_test_split( X, y, test_size=0.2, random_state=42, stratify=y )


vectorizer = TfidfVectorizer(stop_words='english', max_features=10000, ngram_range=(1,2))

x_train_trans = vectorizer.fit_transform(x_train)
x_test_trans = vectorizer.transform(x_test)

In [71]:
model = LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced', C=5.0)
# 규제의 강도 C가 클수록 규제 약함 => 모델 복잠함 => 과적합 위험
model.fit(x_train_trans, y_train)
y_pred = model.predict(x_test_trans)
print(accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

0.6605187319884727
              precision    recall  f1-score   support

        ENFJ       0.57      0.63      0.60        38
        ENFP       0.65      0.61      0.63       135
        ENTJ       0.51      0.52      0.52        46
        ENTP       0.55      0.60      0.57       137
        ESFJ       0.38      0.33      0.35         9
        ESFP       0.00      0.00      0.00        10
        ESTJ       0.75      0.38      0.50         8
        ESTP       0.55      0.61      0.58        18
        INFJ       0.74      0.67      0.70       294
        INFP       0.73      0.75      0.74       366
        INTJ       0.64      0.64      0.64       218
        INTP       0.67      0.75      0.71       261
        ISFJ       0.55      0.52      0.53        33
        ISFP       0.53      0.48      0.50        54
        ISTJ       0.60      0.51      0.55        41
        ISTP       0.63      0.70      0.66        67

    accuracy                           0.66      1735
   macr

In [63]:
df['posts'] = df['posts'].apply(clean_text) # apply : 모든 데이터에 clean_text 일괄 적용

X = df['posts']
y = df['type']

df['EI'] = df['type'].apply(lambda x : x[0])
df['NS'] = df['type'].apply(lambda x : x[1])
df['TF'] = df['type'].apply(lambda x : x[2])
df['JP'] = df['type'].apply(lambda x : x[3])

print(df[['type', 'EI', 'NS', 'TF', 'JP']].head())

   type EI NS TF JP
0  INFJ  I  N  F  J
1  ENTP  E  N  T  P
2  INTP  I  N  T  P
3  INTJ  I  N  T  J
4  ENTJ  E  N  T  J


In [ ]:
def feature_change(feature):
    
    X = df['posts']
    y = df[feature]

    x_train,x_test,y_train,y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

    vectorizer = TfidfVectorizer(stop_words='english', max_features=5000, ngram_range=(1,2))

    x_train_trans = vectorizer.fit_transform(x_train)
    x_test_trans = vectorizer.transform(x_test)

    model = LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced', C=5.0)
    model.fit(x_train_trans, y_train)
    y_pred = model.predict(x_test_trans)
    return model, vectorizer, x_test_trans, y_test, y_pred


In [74]:
models = {}
for c in ['EI', 'NS', 'TF', 'JP']:
    models[c] = feature_change(c)

In [81]:
new_y_pred = [ei+ns+tf+jp for ei, ns, tf, jp in
               zip(models['EI'][4], models['NS'][4], models['TF'][4], models['JP'][4])]
print(classification_report(y_test, new_y_pred))

              precision    recall  f1-score   support

        ENFJ       0.01      0.03      0.02        38
        ENFP       0.13      0.10      0.12       135
        ENTJ       0.03      0.04      0.03        46
        ENTP       0.07      0.07      0.07       137
        ESFJ       0.00      0.00      0.00         9
        ESFP       0.00      0.00      0.00        10
        ESTJ       0.00      0.00      0.00         8
        ESTP       0.00      0.00      0.00        18
        INFJ       0.18      0.14      0.16       294
        INFP       0.20      0.19      0.19       366
        INTJ       0.11      0.10      0.11       218
        INTP       0.15      0.18      0.16       261
        ISFJ       0.02      0.03      0.02        33
        ISFP       0.02      0.02      0.02        54
        ISTJ       0.03      0.02      0.03        41
        ISTP       0.02      0.01      0.02        67

    accuracy                           0.12      1735
   macro avg       0.06   

In [ ]:
print(models['EI'][3].index.equals(models['NS'][3].index))
print(models['EI'][3].index.equals(models['TF'][3].index))
print(models['EI'][3].index.equals(models['JP'][3].index))

In [82]:
def feature_change(feature, x_train, x_test):
    y = df[feature]
    
    # x_train/x_test 인덱스에 맞게 y 뽑기
    y_train = y[x_train.index]
    y_test = y[x_test.index]
    
    # TF-IDF 벡터화
    vectorizer = TfidfVectorizer(stop_words='english', max_features=5000, ngram_range=(1,2))
    x_train_trans = vectorizer.fit_transform(x_train)
    x_test_trans = vectorizer.transform(x_test)
    
    # 모델 생성 & 학습
    model = LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced', C=5.0)
    model.fit(x_train_trans, y_train)
    
    # 예측
    y_pred = model.predict(x_test_trans)
    
    return model, vectorizer, x_test_trans, y_test, y_pred

# 인덱스 한 번만 나누기
x_train, x_test, _, _ = train_test_split(X, df['type'], test_size=0.2, random_state=42, stratify=df['type'])

# 4개 모델 학습
models = {}
for c in ['EI', 'NS', 'TF', 'JP']:
    models[c] = feature_change(c, x_train, x_test)

In [83]:
new_y_pred = [ei+ns+tf+jp for ei, ns, tf, jp in
               zip(models['EI'][4], models['NS'][4], models['TF'][4], models['JP'][4])]
print(classification_report(y_test, new_y_pred))

              precision    recall  f1-score   support

        ENFJ       0.18      0.26      0.22        38
        ENFP       0.50      0.49      0.50       135
        ENTJ       0.23      0.35      0.28        46
        ENTP       0.48      0.42      0.45       137
        ESFJ       0.27      0.33      0.30         9
        ESFP       0.00      0.00      0.00        10
        ESTJ       0.08      0.12      0.10         8
        ESTP       0.18      0.33      0.23        18
        INFJ       0.64      0.56      0.60       294
        INFP       0.67      0.63      0.65       366
        INTJ       0.48      0.50      0.49       218
        INTP       0.63      0.60      0.62       261
        ISFJ       0.27      0.33      0.30        33
        ISFP       0.34      0.33      0.34        54
        ISTJ       0.31      0.29      0.30        41
        ISTP       0.38      0.43      0.41        67

    accuracy                           0.51      1735
   macro avg       0.35   